# Statevector testing utilities

In [ ]:
from guppylang.std.quantum import (
    discard_array,
    discard,
    x,
    ry, 
    cx,
    qubit,
    h
)
from guppylang.std.angles import angle
from guppylang import guppy
from guppylang.std.debug import state_output
from guppylang.std.builtins import comptime, array, nat
from guppyalgos.utils import qarray
import numpy as np

## `get_statevector`
- Measured qubits need to have `state_output("reqresult_state", state)`

In [2]:
n_qubits = 4
@guppy
def main_sv() -> None:
    state = qarray(n_qubits)

    for i in range(n_qubits):
        ry(state[i], angle(3.14/2))

    state_output("result_state", state)
    discard_array(state)

In [3]:
from guppyalgos.tests.helpers import get_statevector, get_unitary

In [4]:
get_statevector(main_sv, n_qubits)

array([ 0.37096824-0.00000000e+00j, -0.2972016 +1.81983495e-17j,
       -0.2972016 +1.81983495e-17j,  0.23810338-2.91592544e-17j,
       -0.2972016 +1.81983495e-17j,  0.23810338-2.91592544e-17j,
        0.23810338-2.91592544e-17j, -0.19075678+3.50414518e-17j,
       -0.2972016 +1.81983495e-17j,  0.23810338-2.91592544e-17j,
        0.23810338-2.91592544e-17j, -0.19075678+3.50414518e-17j,
        0.23810338-2.91592544e-17j, -0.19075678+3.50414518e-17j,
       -0.19075678+3.50414518e-17j,  0.152825  -3.74313289e-17j])

## `get_unitary`
- Measures all qubits
- No result state needed
- Implicit, however require an single qubit register input

In [5]:
@guppy
def main_unitary[n_s: nat](state: array[qubit, n_s]) -> None:

    for i in range(n_qubits):
        ry(state[i], angle(3.14/2))


In [6]:
get_unitary(main_unitary, n_qubits)

array([[ 0.37096824+0.00000000e+00j,  0.2972016 +2.27152533e-17j,
         0.2972016 +2.27152533e-17j,  0.23810338+3.63966991e-17j,
         0.2972016 +2.27152533e-17j,  0.23810338+3.63966991e-17j,
         0.23810338+3.63966991e-17j,  0.19075678+4.37388816e-17j,
         0.2972016 +2.27152533e-17j,  0.23810338+3.63966991e-17j,
         0.23810338+3.63966991e-17j,  0.19075678+4.37388816e-17j,
         0.23810338+3.63966991e-17j,  0.19075678+4.37388816e-17j,
         0.19075678+4.37388816e-17j,  0.152825  +4.67219358e-17j],
       [-0.2972016 +1.81983495e-17j,  0.37096824-4.09136029e-17j,
        -0.23810338-3.61872232e-18j,  0.2972016 -1.00627234e-17j,
        -0.23810338-3.61872232e-18j,  0.2972016 -1.00627234e-17j,
        -0.19075678-1.74787705e-17j,  0.23810338+1.01365879e-17j,
        -0.23810338-3.61872232e-18j,  0.2972016 -1.00627234e-17j,
        -0.19075678-1.74787705e-17j,  0.23810338+1.01365879e-17j,
        -0.19075678-1.74787705e-17j,  0.23810338+1.01365879e-17j,
        -

### Internally allocated work qubits

When a Guppy circuit allocates work qubits internally, pass their count through `n_extra_qubits` so `get_unitary` reserves enough emulator qubits. The work qubits must be returned to $|0\rangle$ before they are discarded.

In [ ]:
from typing import no_type_check

@guppy
@no_type_check
def unitary_with_work_qubit(state: array[qubit, 1]) -> None:
    work = qubit()
    cx(state[0], work)
    cx(state[0], work)
    x(state[0])
    discard(work)

unitary = get_unitary(
    unitary_with_work_qubit,
    n_qubits=1,
    n_extra_qubits=1,
)

expected = np.array([[0, 1], [1, 0]])
assert np.allclose(unitary / unitary[0, 1], expected)
unitary

#### `get_unitary_assumed_phase`

`get_unitary_assumed_phase` accepts `n_extra_qubits` in the same way. Since this helper does not retain phase information, compare the absolute values of the resulting matrix.

In [ ]:
from guppyalgos.tests.helpers import get_unitary_assumed_phase

unitary_assumed_phase = get_unitary_assumed_phase(
    unitary_with_work_qubit,
    n_qubits=1,
    n_extra_qubits=1,
)

assert np.allclose(np.abs(unitary_assumed_phase), expected)
unitary_assumed_phase

## `get_statevector_projected`
- postselect on certain measurement outcomes
- The post selected qubits are the ones measured via `state_output("projection", state)` 
- The rest of the qubits are returned as the output statevector
- Optional renormalization of the output statevector

In [7]:
n_state_qubits = 2
n_ancilla_qubits = 3
n_total_qubits = n_state_qubits + n_ancilla_qubits
@guppy
def main_sv_project() -> None:
    state = qarray(n_state_qubits)
    ancilla = qarray(n_ancilla_qubits)

    for i in range(n_state_qubits):
        ry(state[i], angle(3.14/2))

    for i in range(n_ancilla_qubits):
        ry(ancilla[i], angle(3.14/2))

    for i in range(n_ancilla_qubits):
        for j in range(n_state_qubits):
            cx(state[j], ancilla[i])

    # projection qubits to be measured
    state_output("projection", ancilla)
    state_output("system", state)
    
    discard_array(state)
    discard_array(ancilla)

In [8]:
from guppyalgos.tests.helpers import get_statevector_projected

post_selection = {"projection": [False for _ in range(n_ancilla_qubits)]} # postselect on all ancillas being 0
get_statevector_projected(main_sv_project, n_total_qubits, post_selection, renormalize=True)

array([0.7556292 -5.07091231e-16j, 0.31129088+1.81104011e-17j,
       0.31129088+5.43312033e-17j, 0.48499534+2.17324813e-16j])

## Post selection onto multiple registers with `get_statevector_projected`

- We can project sequentially onto multiple registers, this allows us to project onto a sequence of post selection registers, which is useful for testing algorithms that involve projection onto multiple registers such as QSVT-based algorithms.

In [9]:
n_state_qubits = 2
n_phase_qubits = 3
n_total_qubits = n_state_qubits + n_phase_qubits + 1
state_preparation_angle = np.pi / 4

@guppy
def main() -> None:
    phase_qreg = qarray(n_phase_qubits)
    state_qreg = qarray(n_state_qubits)
    ancilla_qreg = qarray(1)

    # Non-eigenstate input in computational basis.
    ry(state_qreg[0], angle(state_preparation_angle))
    x(state_qreg[1])

    for i in range(n_phase_qubits):
        h(phase_qreg[i])

    h(ancilla_qreg[0])
    for i in range(n_phase_qubits):

        cx(phase_qreg[i], ancilla_qreg[0])

    for i in range(n_state_qubits):
        cx(state_qreg[i], ancilla_qreg[0])  

    h(ancilla_qreg[0])

    state_output("phase", phase_qreg)
    state_output("ancilla", ancilla_qreg)
    state_output("system", state_qreg)
    discard_array(phase_qreg)
    discard_array(state_qreg)
    discard_array(ancilla_qreg)

- One must prepare a postselection dict

In [10]:
from guppyalgos.tests.helpers import get_statevector_projected, statevector_projected_selene
import numpy as np

postselect_dict = {"ancilla": [False], "phase": [False, False, False]}

get_statevector_projected(main, n_total_qubits, postselect_dict)


array([ 2.02524568e-17-2.17207347e-32j,  5.77861354e-17-7.47527359e-33j,
       -8.70182863e-17-3.30747719e-01j, -3.04173840e-16-9.43719209e-01j])

- If one needs more control we can return a `SeleneQuestState` with specified qubits using `statevector_projected_selene` if one wants to do further projections or tests 

In [11]:
statevector_projected_selene(main, n_total_qubits, postselect_dict, returned_specified_qubits=[3,4])

SubQuestState(state=array([ 2.02524568e-17-2.17207347e-32j,  5.77861354e-17-7.47527359e-33j,
       -8.70182863e-17-3.30747719e-01j, -3.04173840e-16-9.43719209e-01j]), total_qubits=2, specified_qubits=[3, 4])

## `get_unitary_projected`
- Supports one or more projected registers
- The helper records projected registers for post-selection using the dictionary key order
- Pre and post selection dictionaries determine the projected registers and their bitstrings
- Pre-selection is supplied register-by-register using the same dictionary keys

In [12]:
@guppy
def main_unitary_projected[n_a: nat, n_s: nat](ancilla: array[qubit, n_a], state: array[qubit, n_s]) -> None:
    for i in range(len(state)):
        ry(state[i], angle(3.14/2))

    for i in range(len(ancilla)):
        ry(ancilla[i], angle(3.14/2))

    for i in range(len(state)):
        for j in range(len(ancilla)):
            cx(state[i], ancilla[j])

In [13]:
from guppyalgos.tests.helpers import get_unitary_projected

post_selection = {"ancilla": [False for _ in range(n_ancilla_qubits)]}
pre_selection = {"ancilla": [True for _ in range(n_ancilla_qubits)]}
get_unitary_projected(main_unitary_projected, n_state_qubits, post_selection, pre_selection)

array([[ 6.93889390e-17+0.14887239j,  1.42420307e-17+0.11926928j,
         1.42420307e-17+0.11926928j,  3.94574074e-17+0.09555271j],
       [-1.80411242e-16-0.23194517j,  1.28749977e-16+0.28951489j,
        -6.68895356e-17-0.18582312j,  1.39427393e-16+0.23194517j],
       [-1.66533454e-16-0.23194517j, -6.68895356e-17-0.18582312j,
         2.67527855e-16+0.28951489j,  1.53305181e-16+0.23194517j],
       [-6.93889390e-18+0.09555271j, -7.30313681e-18-0.11926928j,
        -7.30313681e-18-0.11926928j, -3.72795415e-17+0.14887239j]])

### Internally allocated work qubits

When a Guppy circuit allocates ancilla or work qubits internally, they are not visible in its function signature. Pass their count through `n_extra_qubits` so `get_unitary_projected` reserves enough emulator qubits while extracting only the logical state-register unitary. Internal work qubits must be returned to $|0\rangle$ before they are discarded.

In [ ]:
from typing import no_type_check
from guppyalgos.tests.helpers import get_unitary_projected

@guppy
@no_type_check
def projected_unitary_with_work_qubit(
    projection: array[qubit, 1],
    state: array[qubit, 1],
) -> None:
    work = qubit()
    cx(state[0], work)
    cx(state[0], work)
    x(state[0])
    discard(work)

projected = get_unitary_projected(
    projected_unitary_with_work_qubit,
    n_state_qubits=1,
    post_select_dict={"projection": [False]},
    n_extra_qubits=1,
)

expected = np.array([[0, 1], [1, 0]])
assert np.allclose(projected / projected[0, 1], expected)
projected

### `get_total_state_on_only_specified_registers`

Get the total state only on registers specified by `state_result`s, useful for only looking at the state on non-ancilla qubits. 

Assumes that the specified registers are not entangled with anything else.

The state results should not be separated by any computation.

In [14]:
from guppyalgos.utils import transversal
from selene_sim import Quest
from guppyalgos.tests.helpers import get_total_state_on_only_specified_registers

@guppy
def main_get_total_state_on_only_specified_registers() -> None:
    control_reg = qarray(3)
    target_reg = qarray(3)
    ancilla = qubit()
    x(control_reg[1])
    transversal(cx, control_reg, target_reg)
    # state results for desired registers
    state_output("control", control_reg)
    state_output("target", target_reg)
    discard_array(control_reg)
    discard_array(target_reg)
    discard(ancilla)

res = main_get_total_state_on_only_specified_registers.emulator(7).run()
states = Quest.extract_states_dict(res.results[0].entries)

# returns the state and a dictionary mapping from the tags to the qubit ids for each register
non_ancilla_state, spec_qubit_dict = get_total_state_on_only_specified_registers(
    states, ["target", "control"]
)
control_qubits = spec_qubit_dict["control"]
target_qubits = spec_qubit_dict["target"]

# length of non_ancilla_state will exclude the ancilla, so be 2**6=64,
print(len(non_ancilla_state.state)) 

64


## Use of `QuantumReplay` simulator

The `QuantumReplay` simulator in selene allows for the results of measurements to be specified in advance, using mid-program projection (so long as the overlap with the resulting state is not negligible). This is useful for reducing flakiness in tests that involve mid-circuit measurement or measurement-and-feedforward

For example the following simplified version of a repeat until success rotation via teleportation will continue using rotation resource states for corrections until a success flag is measured. Any number of failures can easily be tested by using the replay simulator.

In [ ]:
from selene_sim import QuantumReplay
from guppylang.std.quantum import measure, h, rz, s, sdg
import numpy as np

@guppy
def repeat_until_success_ry_example() -> None:
    """Perform repeat-until-success Ry rotation."""
    q = qubit()
    sdg(q)
    h(q)
    theta = angle(0.1)
    attempts = 0
    while True:
        attempts += 1
        a = qubit()
        h(a)
        rz(a, theta * (2 ** (attempts - 1)))
        cx(q, a)
        if not measure(a).read():
            continue
        break
    h(q)
    s(q)
    state_output("result_state",q)
    discard(q)

# set up the replay plugin with desired measurements and Quest as a backend
n_shots = 2
rus_replay_sim = QuantumReplay(simulator=Quest(), measurements=[[True], [False, False, True]])


em_result = (
        repeat_until_success_ry_example.emulator(2).with_simulator(rus_replay_sim).with_shots(n_shots).run()
    )
# print the rotation angle applied in each case
for shot_result in em_result.results:
    states = Quest.extract_states_dict(shot_result)
    sv = states["result_state"].state
    res = abs(sv[3]) ** 2
    theta = 2 * np.arcsin(np.sqrt(res)) / np.pi
    print(theta)


0.10000000000000005
0.10000000000000002
